# Retail Data Integration — Data Inspection

## Objective

This notebook inspects the raw Indian FMCG retail dataset before cleaning.  
The purpose is to understand its structure, data types, missing values, record grain, and potential quality issues without modifying the source file.

## 1. Import Libraries and Load the Dataset

Pandas is used to load and inspect the CSV file. The raw source file is read without applying any cleaning or transformations.

In [5]:
import pandas as pd

file_path = r"C:\Users\janakiram\Documents\Data_Analytics_Portfolio\PROJECTS\Retail_Data_Integration\data\raw\Indian FMCG Retail Sales  Customer  Inventory (2024).csv"

df = pd.read_csv(file_path)

## 2. Dataset Overview

This section checks the dataset dimensions, sample records, column names, data types, and non-null counts.

In [6]:
print("Rows and columns:", df.shape)

display(df.head())

print("\nColumn information:")
df.info()

Rows and columns: (100000, 21)


,Invoice_ID,Invoice_Date,City,Store_Format,Category,Brand,Channel,Payment_Mode,Units,Cost_Price,...,Revenue,Cost,Margin,Margin_%,Stock_On_Hand,Reorder_Level,Lead_Time_Days,Customer_Age,Customer_Gender,Loyalty_Flag
0,58018430,2024-02-02 13:50:00,Kolkata,Super,Grocery,Nestle,Online,Wallet,2,148.999035,...,350.114418,297.998070,52.116348,0.148855,154,31,11,20.0,M,1
1,48157952,2024-10-09 11:52:00,Hyderabad,Hyper,Home Care,PepsiCo,Offline,Wallet,1,80.626759,...,106.034197,80.626759,25.407438,0.239616,130,24,5,26.0,F,1
2,23283831,2024-08-26 22:03:00,Chennai,Hyper,Snacks,PepsiCo,Online,Card,5,181.233170,...,1044.900114,906.165848,138.734266,0.132773,270,44,8,27.0,F,0
3,53537460,2024-06-09 04:34:00,Bengaluru,Hyper,Beverages,Nestle,Omnichannel,Card,2,195.811402,...,521.759920,391.622805,130.137115,0.249420,403,63,8,NaN,F,0
4,55348596,2024-06-07 01:13:00,Delhi,Super,Snacks,ITC,Offline,Card,3,23.571888,...,91.873416,70.715665,21.157751,0.230292,366,40,9,60.0,F,1



Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Invoice_ID       100000 non-null  int64  
 1   Invoice_Date     100000 non-null  object 
 2   City             100000 non-null  object 
 3   Store_Format     100000 non-null  object 
 4   Category         100000 non-null  object 
 5   Brand            100000 non-null  object 
 6   Channel          100000 non-null  object 
 7   Payment_Mode     100000 non-null  object 
 8   Units            100000 non-null  int64  
 9   Cost_Price       100000 non-null  float64
 10  Selling_Price    100000 non-null  float64
 11  Revenue          100000 non-null  float64
 12  Cost             100000 non-null  float64
 13  Margin           100000 non-null  float64
 14  Margin_%         100000 non-null  float64
 15  Stock_On_Hand    100000 non-null  int64  
 16  Reorder_Level    1

## 3. Invoice ID and Record-Grain Investigation

A unique invoice ID may represent either one complete transaction or multiple product lines within the same invoice. Repeated invoice IDs are investigated before deciding whether they are valid line items or duplicate records.

In [7]:
invoice_counts = df["Invoice_ID"].value_counts()

print("Total rows:", len(df))
print("Unique invoice IDs:", df["Invoice_ID"].nunique())
print("Invoice IDs appearing more than once:", (invoice_counts > 1).sum())
print("Maximum rows under one invoice ID:", invoice_counts.max())

repeated_invoices = df[
    df.duplicated(subset="Invoice_ID", keep=False)
].sort_values("Invoice_ID")

display(repeated_invoices.head(20))

print("Exact duplicate rows:", df.duplicated().sum())

Total rows: 100000
Unique invoice IDs: 99938
Invoice IDs appearing more than once: 62
Maximum rows under one invoice ID: 2


,Invoice_ID,Invoice_Date,City,Store_Format,Category,Brand,Channel,Payment_Mode,Units,Cost_Price,...,Revenue,Cost,Margin,Margin_%,Stock_On_Hand,Reorder_Level,Lead_Time_Days,Customer_Age,Customer_Gender,Loyalty_Flag
44817,10626453,2024-05-24 08:36:00,Hyderabad,Super,Personal Care,Parle,Omnichannel,Card,1,194.537132,...,210.061916,194.537132,15.524784,0.073906,274,24,7,NaN,F,0
7231,10626453,2024-05-08 17:34:00,Ahmedabad,Super,Dairy,Nestle,Omnichannel,Wallet,3,127.292978,...,493.333495,381.878934,111.454561,0.225921,289,63,6,30.0,F,1
33690,11762171,2024-02-28 11:12:00,Bengaluru,Super,Beverages,Tata,Omnichannel,Card,3,96.824572,...,337.815246,290.473715,47.341530,0.140140,115,54,3,NaN,F,0
97489,11762171,2024-07-03 10:49:00,Bengaluru,Super,Vegetables,HUL,Offline,Card,1,145.610215,...,185.530928,145.610215,39.920714,0.215170,324,50,10,53.0,F,0
68639,14111027,2024-10-04 00:35:00,Delhi,Hyper,Snacks,Britannia,Online,UPI,4,182.849529,...,868.842934,731.398117,137.444817,0.158193,195,71,8,18.0,F,0
49989,14111027,2024-12-27 05:33:00,Bengaluru,Express,Personal Care,Parle,Omnichannel,UPI,2,83.063733,...,239.581590,166.127466,73.454124,0.306593,125,48,13,NaN,F,1
35276,15435618,2024-12-12 10:18:00,Pune,Super,Snacks,PepsiCo,Online,UPI,2,118.223407,...,337.503707,236.446814,101.056893,0.299425,392,73,9,NaN,M,1
84363,15435618,2024-05-07 14:37:00,Bengaluru,Super,Dairy,Tata,Online,Wallet,2,15.117169,...,41.983548,30.234339,11.749209,0.279853,343,63,10,55.0,NaN,0
37082,18451050,2024-10-25 21:58:00,Pune,Hyper,Beverages,Tata,Online,Wallet,3,24.817759,...,84.044396,74.453277,9.591118,0.114120,389,46,14,NaN,M,0
89113,18451050,2024-03-10 11:35:00,Chennai,Super,Vegetables,Amul,Offline,Card,1,105.377459,...,114.990964,105.377459,9.613505,0.083602,432,68,14,NaN,M,0


Exact duplicate rows: 0


### Invoice ID Investigation — Finding

The dataset contains 100,000 transaction rows but only 99,938 unique invoice IDs. A total of 62 invoice IDs appear twice.

Rows sharing an invoice ID have different dates, locations, products, and transaction values. They are therefore not conventional multi-line invoices. No completely identical records were found.

The repeated IDs are treated as source-system identifier collisions. Both records will be retained, and a surrogate `Transaction_ID` will be created during data cleaning to provide a reliable unique key.

## 4. Missing-Value Analysis

Missing values are measured by count and percentage. Blank text values are also checked separately because they may not be recognized as null values by pandas.

No values are filled or removed during this inspection stage.

In [9]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

display(missing_summary)

,Missing_Count,Missing_Percentage
Customer_Age,40081,40.08
Customer_Gender,5048,5.05


### Blank-Text Check

Text columns are checked for empty strings and values containing only spaces.

In [11]:
blank_summary = pd.DataFrame({
    "Blank_Count": [
        (
            df[column].notna()
            & df[column].astype(str).str.strip().eq("")
        ).sum()
        for column in text_columns
    ]
}, index=text_columns)

blank_summary = blank_summary[
    blank_summary["Blank_Count"] > 0
].sort_values("Blank_Count", ascending=False)

display(blank_summary)

,Blank_Count


### Missing-Value Overlap

The overlap between missing customer age and gender is checked to determine whether the two fields are missing independently or as part of the same customer-profile issue.

In [12]:
age_missing = df["Customer_Age"].isna()
gender_missing = df["Customer_Gender"].isna()

missing_patterns = pd.Series({
    "Both age and gender missing": (age_missing & gender_missing).sum(),
    "Only age missing": (age_missing & ~gender_missing).sum(),
    "Only gender missing": (~age_missing & gender_missing).sum(),
    "Neither missing": (~age_missing & ~gender_missing).sum()
})

missing_patterns = missing_patterns.to_frame("Row_Count")
missing_patterns["Percentage"] = (
    missing_patterns["Row_Count"] / len(df) * 100
).round(2)

display(missing_patterns)

,Row_Count,Percentage
Both age and gender missing,2059,2.06
Only age missing,38022,38.02
Only gender missing,2989,2.99
Neither missing,56930,56.93


### Missing-Value Findings

`Customer_Age` has a high missing rate of 40.08%, while `Customer_Gender` is missing in 5.05% of records.

Only 2.06% of rows are missing both values. Most missing-age records still contain gender information, indicating that the two fields are not always missing for the same reason.

No imputation is performed during inspection. Age and gender will be handled separately during cleaning to avoid introducing unsupported customer information.

## 5. Categorical-Value Inspection

Categorical columns are reviewed for unexpected values, inconsistent spelling, capitalization differences, whitespace, and invalid category codes.

The values are inspected without modifying the dataset.

In [14]:
categorical_columns = [
    "City",
    "Store_Format",
    "Category",
    "Brand",
    "Channel",
    "Payment_Mode",
    "Customer_Gender",
    "Loyalty_Flag"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Distinct non-null values:", df[column].nunique())
    print(df[column].value_counts(dropna=False))


--- City ---
Distinct non-null values: 8
City
Kolkata      12681
Delhi        12649
Mumbai       12492
Chennai      12467
Hyderabad    12440
Ahmedabad    12436
Bengaluru    12432
Pune         12403
Name: count, dtype: int64

--- Store_Format ---
Distinct non-null values: 3
Store_Format
Hyper      33443
Express    33314
Super      33243
Name: count, dtype: int64

--- Category ---
Distinct non-null values: 8
Category
Fruits           12708
Home Care        12630
Beverages        12585
Snacks           12514
Vegetables       12472
Grocery          12412
Personal Care    12409
Dairy            12270
Name: count, dtype: int64

--- Brand ---
Distinct non-null values: 8
Brand
ITC          12649
HUL          12572
Nestle       12560
Amul         12521
PepsiCo      12507
Britannia    12497
Tata         12461
Parle        12233
Name: count, dtype: int64

--- Channel ---
Distinct non-null values: 3
Channel
Omnichannel    33383
Offline        33362
Online         33255
Name: count, dtype: int64



In [16]:
text_columns = df.select_dtypes(include="object").columns

whitespace_summary = pd.DataFrame({
    "Values_With_Outer_Spaces": [
        (
            df[column].notna()
            & df[column].ne(df[column].str.strip())
        ).sum()
        for column in text_columns
    ]
}, index=text_columns)

display(
    whitespace_summary[
        whitespace_summary["Values_With_Outer_Spaces"] > 0
    ]
)

,Values_With_Outer_Spaces


### Categorical-Value Findings

The categorical columns contain consistent labels with no visible spelling or capitalization variations.

`Customer_Gender` contains `M`, `F`, and `O`, with null values treated separately. `Loyalty_Flag` contains only the valid binary values `0` and `1`.

The category frequencies are relatively balanced, with no single value unexpectedly dominating its field.

## 6. Numerical-Value Inspection

Numerical columns are summarized and checked for invalid or unusual values. Business-valid conditions, such as inventory falling below the reorder level, are measured but are not automatically treated as errors.

In [18]:
numeric_columns = [
    "Units",
    "Cost_Price",
    "Selling_Price",
    "Revenue",
    "Cost",
    "Margin",
    "Margin_%",
    "Stock_On_Hand",
    "Reorder_Level",
    "Lead_Time_Days",
    "Customer_Age"
]

numeric_summary = df[numeric_columns].describe().T
display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
Units,100000.0,2.998290,1.414619,1.000000,2.000000,3.000000,4.000000,5.000000
Cost_Price,100000.0,104.859625,54.825341,10.000526,57.478508,104.903700,152.320122,199.999659
Selling_Price,100000.0,131.107353,69.862379,10.580915,71.327813,129.999158,188.854907,289.699598
Revenue,100000.0,393.351350,297.154653,10.653198,155.415176,313.349100,573.063568,1443.134985
Cost,100000.0,314.614493,234.901995,10.002024,125.467072,252.801900,462.376451,999.985219
Margin,100000.0,78.736857,74.236348,0.556870,24.448402,53.552720,110.075103,447.207497
Margin_%,100000.0,0.193368,0.075143,0.047621,0.131251,0.200287,0.259151,0.310343
Stock_On_Hand,100000.0,274.156670,129.853861,50.000000,161.000000,274.000000,387.000000,499.000000
Reorder_Level,100000.0,49.467710,17.305614,20.000000,34.000000,50.000000,64.000000,79.000000
Lead_Time_Days,100000.0,8.516810,3.458274,3.000000,6.000000,9.000000,12.000000,14.000000


In [19]:
numeric_checks = pd.Series({
    "Units <= 0": (df["Units"] <= 0).sum(),
    "Cost Price <= 0": (df["Cost_Price"] <= 0).sum(),
    "Selling Price <= 0": (df["Selling_Price"] <= 0).sum(),
    "Revenue < 0": (df["Revenue"] < 0).sum(),
    "Cost < 0": (df["Cost"] < 0).sum(),
    "Margin < 0": (df["Margin"] < 0).sum(),
    "Margin % < 0": (df["Margin_%"] < 0).sum(),
    "Margin % > 1": (df["Margin_%"] > 1).sum(),
    "Stock on Hand < 0": (df["Stock_On_Hand"] < 0).sum(),
    "Reorder Level < 0": (df["Reorder_Level"] < 0).sum(),
    "Lead Time Days <= 0": (df["Lead_Time_Days"] <= 0).sum(),
    "Customer Age < 0": (df["Customer_Age"] < 0).sum(),
    "Customer Age > 120": (df["Customer_Age"] > 120).sum(),
    "Stock at or below Reorder Level": (
        df["Stock_On_Hand"] <= df["Reorder_Level"]
    ).sum()
})

display(numeric_checks.to_frame("Row_Count"))

,Row_Count
Units <= 0,0
Cost Price <= 0,0
Selling Price <= 0,0
Revenue < 0,0
Cost < 0,0
Margin < 0,0
Margin % < 0,0
Margin % > 1,0
Stock on Hand < 0,0
Reorder Level < 0,0


In [20]:
non_integer_ages = (
    df["Customer_Age"].dropna() % 1 != 0
).sum()

print("Non-integer customer ages:", non_integer_ages)

Non-integer customer ages: 0


### Numerical-Value Findings

All numerical columns fall within plausible ranges, and no negative or zero values were found where positive values are expected.

Customer ages are whole numbers between 18 and 64. Margin percentages range from approximately 4.76% to 31.03%.

A total of 1,729 records have stock at or below their reorder level. These records are valid inventory conditions and may require replenishment rather than data correction.

## 7. Calculated-Field Validation

Revenue, cost, margin, and margin percentage are checked against their expected business formulas.

Small floating-point differences are tolerated using NumPy's `isclose` function.

In [21]:
import numpy as np

expected_revenue = df["Units"] * df["Selling_Price"]
expected_cost = df["Units"] * df["Cost_Price"]
expected_margin = df["Revenue"] - df["Cost"]
expected_margin_percentage = df["Margin"] / df["Revenue"]

calculation_checks = pd.Series({
    "Revenue mismatches": (
        ~np.isclose(df["Revenue"], expected_revenue, atol=0.01)
    ).sum(),

    "Cost mismatches": (
        ~np.isclose(df["Cost"], expected_cost, atol=0.01)
    ).sum(),

    "Margin mismatches": (
        ~np.isclose(df["Margin"], expected_margin, atol=0.01)
    ).sum(),

    "Margin percentage mismatches": (
        ~np.isclose(
            df["Margin_%"],
            expected_margin_percentage,
            atol=0.0001
        )
    ).sum(),

    "Selling price below cost price": (
        df["Selling_Price"] < df["Cost_Price"]
    ).sum()
})

display(calculation_checks.to_frame("Row_Count"))

,Row_Count
Revenue mismatches,0
Cost mismatches,0
Margin mismatches,0
Margin percentage mismatches,0
Selling price below cost price,0


In [22]:
calculation_differences = pd.Series({
    "Maximum revenue difference": (
        df["Revenue"] - expected_revenue
    ).abs().max(),

    "Maximum cost difference": (
        df["Cost"] - expected_cost
    ).abs().max(),

    "Maximum margin difference": (
        df["Margin"] - expected_margin
    ).abs().max(),

    "Maximum margin percentage difference": (
        df["Margin_%"] - expected_margin_percentage
    ).abs().max()
})

display(calculation_differences.to_frame("Maximum_Absolute_Difference"))

,Maximum_Absolute_Difference
Maximum revenue difference,6.821210e-13
Maximum cost difference,4.547474e-13
Maximum margin difference,3.979039e-13
Maximum margin percentage difference,2.220446e-16


### Calculated-Field Findings

Revenue, cost, margin, and margin percentage match their expected business formulas for all 100,000 records.

The extremely small maximum differences are caused by normal floating-point precision and are not data-quality issues. The existing calculated fields are therefore considered reliable.

## 8. Invoice-Date Validation

`Invoice_Date` was initially imported as text. A temporary datetime conversion is performed to identify invalid values and confirm the reporting period before permanently changing the data type.

In [23]:
parsed_invoice_date = pd.to_datetime(
    df["Invoice_Date"],
    errors="coerce"
)

date_checks = pd.Series({
    "Invalid dates": parsed_invoice_date.isna().sum(),
    "Dates before 2024": (parsed_invoice_date.dt.year < 2024).sum(),
    "Dates after 2024": (parsed_invoice_date.dt.year > 2024).sum(),
    "Duplicate timestamps": parsed_invoice_date.duplicated().sum()
})

display(date_checks.to_frame("Row_Count"))

print("Earliest invoice:", parsed_invoice_date.min())
print("Latest invoice:", parsed_invoice_date.max())

,Row_Count
Invalid dates,0
Dates before 2024,0
Dates after 2024,0
Duplicate timestamps,8943


Earliest invoice: 2024-01-01 00:00:00
Latest invoice: 2024-12-30 23:57:00


In [24]:
monthly_counts = (
    parsed_invoice_date
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename_axis("Invoice_Month")
    .to_frame("Transaction_Count")
)

display(monthly_counts)

,Transaction_Count
Invoice_Month,
2024-01,8514
2024-02,7944
2024-03,8474
2024-04,8287
2024-05,8468
2024-06,8095
2024-07,8497
2024-08,8524
2024-09,8316


In [25]:
time_components = pd.DataFrame({
    "Hour": parsed_invoice_date.dt.hour,
    "Minute": parsed_invoice_date.dt.minute,
    "Second": parsed_invoice_date.dt.second
})

print(
    "Rows containing a time component:",
    (
        (time_components["Hour"] != 0)
        | (time_components["Minute"] != 0)
        | (time_components["Second"] != 0)
    ).sum()
)

Rows containing a time component: 99925


### Invoice-Date Findings

All invoice timestamps are valid and fall within 2024. Every month is represented in the dataset.

A total of 8,943 timestamps repeat, but repeated timestamps do not indicate duplicate transactions because multiple retail transactions can occur at the same time.

The `Invoice_Date` field will be converted from text to datetime during cleaning, while preserving both its date and time components.